# Generative Adversarial Network (GAN) (2-27-26)




## Pytorch Implementation

In [ ]:
#############
## IMPORTS ##
#############
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.utils import make_grid, save_image

In [ ]:
######################
## DEVICE SELECTION ##
######################
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
###############
## VARIABLES ##
###############
# Just gathering all the variables in one place for easy access and modification.

# Location of the data
data_root = "./data"
# Location to save generated samples
out_dir = "./samples"
# Number of epochs to train
epochs = 20
# Size of the batches during training
batch_size = 128
# Learning rate for optimizer
lr = 2e-4
# Beta1 hyperparameter for Adam optimizers
beta1 = 0.5
# Latent dimension
nz = 100
# Feature maps in Generator
ngf = 64
# Feature maps in Discriminator
ndf = 64

In [ ]:
############################
## MAKE OUTPUT DIRECTORY ##
###########################
os.makedirs(out_dir, exist_ok=True)

In [ ]:
#####################
## IMPORT THE DATA ##
#####################
# We need to transform the data into tensors and normalize it to be in the range [-1, 1] for better training of the GAN.
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),  # maps [0,1] -> [-1,1]
])

# Import the MNIST dataset using torchvision, applying the transformations defined above.
dataset = torchvision.datasets.MNIST(root=data_root, train=True, download=True, transform=transform)

# Create a DataLoader to handle batching and shuffling of the dataset during training.
loader = DataLoader(dataset,batch_size=batch_size,shuffle=True)

In [ ]:
###############
## GENERATOR ##
###############
class Generator(nn.Module):
    """
    The Generator takes a random noise vector z and transforms it into a fake image.
    """
    def __init__(self, nz=100, ngf=64, nc=1):
        """
        Inputs:
            nz: Dimension of the input noise vector.
            ngf: Number of generator feature maps.
            nc: Number of output channels (1 for grayscale images).
        Returns:
            None.
        Initializes the Generator model with a series of ConvTranspose2d layers to upsample the noise vector into a 28x28 image.
        """
        # Call the parent class constructor
        super().__init__()
        # Define the network architecture using nn.Sequential for simplicity.
        self.net = nn.Sequential(
            # Transformation: 1x1 -> 7x7
            nn.ConvTranspose2d(nz, ngf * 4, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # Transformation: 7x7 -> 14x14
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # Transformation: 14x14 -> 28x28
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # refine channels, keep 28x28
            nn.Conv2d(ngf, nc, kernel_size=3, stride=1, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        """
        Inputs:
            z: A batch of noise vectors with shape (N, nz, 1, 1).
        Returns:
            A batch of generated images with shape (N, nc, 28, 28).
        Performs a forward pass through the network, transforming the input noise vector into a generated image.
        """
        return self.net(z)

In [ ]:
###################
## DISCRIMINATOR ##
###################

class Discriminator(nn.Module):
    """
    The Discriminator takes an image x and outputs a logit indicating whether the image is real or fake.
    """
    def __init__(self, ndf=64, nc=1):
        """
        Inputs:
            ndf: Number of discriminator feature maps.
            nc: Number of input channels (1 for grayscale images).
        Returns:
            None.
        Initializes the Discriminator model with a series of Conv2d layers to downsample the input image and 
        produce a logit indicating real/fake. The critical fix is to ensure that the output is always a single 
        scalar per image, regardless of the input image size.
        """
        # Call the parent class constructor
        super().__init__()
        # Define the feature extraction layers using nn.Sequential for simplicity.
        self.features = nn.Sequential(
            # 28x28 -> 14x14
            nn.Conv2d(nc, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 14x14 -> 7x7
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 7x7 -> 4x4  (stride 2 with padding keeps it stable)
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # produces a 1-channel logit map (could be spatial)
        self.logit_head = nn.Conv2d(ndf * 4, 1, kernel_size=3, stride=1, padding=1, bias=False)

        # Define a pooling layer to ensure we get a single logit per image, regardless of the spatial dimensions of the feature map.
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        """
        Inputs:
            x: A batch of images with shape (N, nc, 28, 28).
        Returns:
            A batch of logits with shape (N,).
        Performs a forward pass through the network, transforming the input image into a logit indicating real/fake.
        """
        x = self.features(x)         # (N, C, H, W)
        x = self.logit_head(x)       # (N, 1, H, W)
        x = self.pool(x)             # (N, 1, 1, 1) 
        return x.view(-1)            # (N,)

In [ ]:
###############################
## INITIALIZATION OF WEIGHTS ##
###############################
def weights_init(m):
    """
    Inputs:
        m: A module in the network.
    Returns:
        None.
    Initializes the weights of the network using a normal distribution for convolutional layers and batch normalization layers, 
    following the DCGAN paper's recommendations for stable training.
    """
    # Get the class name of the module to determine how to initialize its weights.
    name = m.__class__.__name__
    # Initialize convolutional layers with a normal distribution centered at 0 with a standard deviation of 0.02.
    if "Conv" in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    # Initialize batch normalization layers with a normal distribution centered at 1 with a standard deviation of 0.02 
    # for weights, and set biases to 0.
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


In [ ]:
############################################
## DEFINE THE GENERATOR AND DISCRIMINATOR ##
############################################
# Define the Generator and Discriminator models, move them to the appropriate device (CPU or GPU), and apply the weight 
# initialization function to ensure stable training.
G = Generator(nz, ngf).to(device)
D = Discriminator(ndf).to(device)
G.apply(weights_init)
D.apply(weights_init)


In [ ]:
#############################################
## DEFINE THE LOSS FUNCTION AND OPTIMIZERS ##
#############################################

criterion = nn.BCEWithLogitsLoss()
optG = optim.Adam(G.parameters(), lr=lr, betas=(beta1, 0.999))
optD = optim.Adam(D.parameters(), lr=lr, betas=(beta1, 0.999))


In [ ]:
##################
## RANDOM NOISE ##
##################
# Create a fixed noise vector for generating consistent samples during training visualization. This allows us to see how the 
# Generator's output evolves over time for the same input noise.
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

In [ ]:
###################
## TRAINING LOOP ##
###################
step = 0
for epoch in range(1, epochs + 1):
    for real, _ in loader:
        real = real.to(device)
        bsz = real.size(0)

        # --- Create label tensors that match D output shape exactly ---
        # D(real) will be shape (bsz,), so use ones_like/zeros_like after computing logits
        # This is a robust pattern for BCEWithLogitsLoss shape safety. [1](https://drdroid.io/stack-diagnosis/pytorch-userwarning--using-a-target-size--torch-size--that-is-different-to-the-input-size--torch-size)[5](https://learn.microsoft.com/en-us/training/modules/introduction-foundry-iq/?WT.mc_id=api_CatalogApi&sso=viva-learning)

        # ---- Train D ----
        D.zero_grad(set_to_none=True)

        logits_real = D(real)                     # (bsz,)
        labels_real = torch.ones_like(logits_real)
        lossD_real = criterion(logits_real, labels_real)

        noise = torch.randn(bsz, cfg.nz, 1, 1, device=cfg.device)
        fake = G(noise).detach()
        logits_fake = D(fake)                     # (bsz,)
        labels_fake = torch.zeros_like(logits_fake)
        lossD_fake = criterion(logits_fake, labels_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optD.step()

        # ---- Train G ----
        G.zero_grad(set_to_none=True)

        noise = torch.randn(bsz, cfg.nz, 1, 1, device=cfg.device)
        fake = G(noise)
        logits = D(fake)                          # (bsz,)
        labels_gen = torch.ones_like(logits)      # want D(fake) -> real
        lossG = criterion(logits, labels_gen)

        lossG.backward()
        optG.step()

        if step % 200 == 0:
            print(f"Epoch [{epoch}/{cfg.epochs}] Step {step} "
                  f"lossD={lossD.item():.4f} lossG={lossG.item():.4f} "
                  f"D(real)={logits_real.shape} D(fake)={logits_fake.shape}")
        step += 1

    # Save sample grid each epoch
    G.eval()
    with torch.no_grad():
        samples = G(fixed_noise).cpu()
    G.train()

    grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
    out_path = os.path.join(cfg.out_dir, f"mnist_dcgan_epoch_{epoch:03d}.png")
    save_image(grid, out_path)
    print(f"Saved samples: {out_path}")

print("Done.")

# This took 85 minutes to run on my Mac but note that it also produces an image with each epoch
# which does take time. You can comment out this block of code if you want to speed up the training
# or you can reduce the number of epochs to 10 or 5 for testing purposes.

/Users/butlerju/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch [1/20] Step 0 lossD=1.4210 lossG=0.6490 D(real)=torch.Size([128]) D(fake)=torch.Size([128])
Epoch [1/20] Step 200 lossD=1.0292 lossG=1.0687 D(real)=torch.Size([128]) D(fake)=torch.Size([128])
Epoch [1/20] Step 400 lossD=0.9280 lossG=0.8033 D(real)=torch.Size([128]) D(fake)=torch.Size([128])
Saved samples: ./samples/mnist_dcgan_epoch_001.png
Epoch [2/20] Step 600 lossD=0.9700 lossG=0.9439 D(real)=torch.Size([128]) D(fake)=torch.Size([128])
Epoch [2/20] Step 800 lossD=1.0710 lossG=1.0188 D(real)=torch.Size([128]) D(fake)=torch.Size([128])
Saved samples: ./samples/mnist_dcgan_epoch_002.png
Epoch [3/20] Step 1000 lossD=1.2629 lossG=2.0697 D(real)=torch.Size([128]) D(fake)=torch.Size([128])
Epoch [3/20] Step 1200 lossD=0.9867 lossG=1.0409 D(real)=torch.Size([128]) D(fake)=torch.Size([128])
Epoch [3/20] Step 1400 lossD=0.9553 lossG=1.2768 D(real)=torch.Size([128]) D(fake)=torch.Size([128])
Saved samples: ./samples/mnist_dcgan_epoch_003.png
Epoch [4/20] Step 1600 lossD=1.5219 lossG=0.59